In [1]:
GRAPH = {
    'S': {'A': 2, 'B': 4, 'C': 3},
    'A': {'D': 4, 'E': 7},
    'B': {'E': 1, 'F': 5},
    'C': {'F': 2},
    'D': {'G': 5},
    'E': {'G': 3},
    'F': {'G': 2},
    'G': {}
}


H1 = {
    'S': 7,
    'A': 7,
    'B': 4,
    'C': 4,
    'D': 5,
    'E': 3,
    'F': 2,
    'G': 0
}


H2 = {
    'S': 7,
    'A': 9,
    'B': 4,
    'C': 5,
    'D': 5,
    'E': 3,
    'F': 2,
    'G': 0
}


In [ ]:
import math, heapq
def a_star_search(graph, h, start, goal):
    g_costs = {node : math.inf for node in graph}
    g_costs[start] = 0
    pq = [(g_costs[start] + h[start], start, 0, [start])]

    expansion_order = []
    while pq:
        f, node, g, path = heapq.heappop(pq)

        if g > g_costs[node]:
            continue

        expansion_order.append(node)
        if goal == node:
            return  path, g, expansion_order

        for neigh in graph[node]:
            new_g = g + graph[node][neigh]
            if new_g < g_costs[neigh]:  
                g_costs[neigh] = new_g
                new_path = path + [neigh]
                heapq.heappush(pq, (new_g + h[neigh], neigh, new_g, new_path))


    return (None, -1, expansion_order)


In [ ]:
def beam_search(graph, h, start, goal, beam_width=2):
    if beam_width <= 0:
        raise ValueError("Beam width must be positive")
    kept_levels = []

    if start == goal:
        return [start], 0, []

    beam = [(start, [start], 0)]
    while beam:
        best_candidate = {}

        for node, path, g in beam:
            for neigh in sorted(graph[node]):
                if neigh in path:
                    continue
                new_g = g + graph[node][neigh]
                candidate = (neigh, path + [neigh] ,new_g)

                if neigh not in best_candidate or new_g < best_candidate[neigh][2]:
                    best_candidate[neigh] = candidate
        if not best_candidate:
            return None, -1, kept_levels

        candidates = sorted(best_candidate.values(), key=lambda item: (h[item[0]], item[0]))
        print(candidates)
        beam = candidates[:beam_width]
        kept_levels.append([node for node, _, _ in beam])
        print("Kept level" , kept_levels)

        for node, path, g in beam:
            if node == goal:
                return path, g, kept_levels

    return None, -1, kept_levels

In [2]:
def path_cost(graph, path):
    if not path:
        return -1
    return sum(graph[u][v] for u, v in zip(path, path[1:]))

In [ ]:

def greedy_best_first_search(graph, h, start, goal):
    """Return (path, cost, expansion_order)."""
    frontier = [(h[start], start, [start])]
    visited = set()
    expansion_order = []

    while frontier:
        _, node, path = heapq.heappop(frontier)

        if node in visited:
            continue

        visited.add(node)
        expansion_order.append(node)

        if node == goal:
            return path, path_cost(graph, path), expansion_order

        for nbr in sorted(graph[node]):
            if nbr not in visited:
                heapq.heappush(frontier, (h[nbr], nbr, path + [nbr]))

    return None, -1, expansion_order


In [4]:
H_IDA = {'S': 4, 'A': 5, 'B': 3, 'C': 2, 'D': 4, 'E': 2, 'F': 1, 'G': 0}


def ida_star(graph, h, start, goal):
    threshold = h[start]
    thersholds = []
    no_of_expansions = 0

    def dfs(path, g, bound):
        nonlocal no_of_expansions
        node = path[-1]
        f = g + h[node]

        if f > bound:
            return None, f

        no_of_expansions += 1

        if node == goal:
            return path, g
        min_exceeded = math.inf
        for neigh in sorted(graph[node]):
            if neigh in path:
                continue
            new_g = g + graph[node][neigh]
            result, exceeded = dfs(path + [neigh], new_g, bound)

            if result is not None:
                return result, exceeded
            min_exceeded = min(exceeded, min_exceeded)

        return None, min_exceeded

    while True:
        thersholds.append(threshold)
        result, next_threshold = dfs([start], 0, threshold)

        if result is not None:
            return result, path_cost(graph, result), thersholds, no_of_expansions

        if next_threshold == math.inf:
            return None, -1, thersholds, no_of_expansions

        threshold = next_threshold

In [6]:
import random, math
N = 8


def queen_conflicts(state):
    conflicts = 0

    for c1 in range(N):
        for c2 in range(c1 + 1, N):
            same_row = state[c1] == state[c2]
            same_diag = abs(state[c1] - state[c2]) == abs(c1 - c2)

            if same_row or same_diag:
                conflicts += 1

    return conflicts


def random_neighbor(state):
    nxt = state.copy()
    col = random.randrange(N)
    old_row = nxt[col]
    choices = [r for r in range(N) if r != old_row]
    nxt[col] = random.choice(choices)
    return nxt


def temperature_schedule(t, T0=10.0, cooling=0.995):
    return T0 * (cooling ** t)


def accept_worse(delta, T):
    if delta <= 0:
        return 1.0

    if T <= 0:
        return 0.0

    return math.exp(-delta / T)


def simulated_annealing(max_steps=5000, seed=0, T0=10.0, cooling=0.995):
    random.seed(seed)

    current = [random.randrange(N) for _ in range(N)]
    current_cost = queen_conflicts(current)

    best = current.copy()
    best_cost = current_cost
    history = [best_cost]

    for t in range(max_steps):
        T = temperature_schedule(t, T0=T0, cooling=cooling)

        if T <= 1e-8 or best_cost == 0:
            break

        candidate = random_neighbor(current)
        candidate_cost = queen_conflicts(candidate)
        delta = candidate_cost - current_cost

        accepted = (
            delta <= 0
            or random.random() < accept_worse(delta, T)
        )

        if accepted:
            current = candidate
            current_cost = candidate_cost

            if current_cost < best_cost:
                best = current.copy()
                best_cost = current_cost

        history.append(best_cost)

    return best, best_cost, history


sa_state, sa_cost, sa_history = simulated_annealing(seed=7)

In [8]:
MAX_PAIRS = N * (N - 1) // 2


def fitness(chromosome):
    return MAX_PAIRS - queen_conflicts(chromosome)


def random_chromosome():
    return [random.randrange(N) for _ in range(N)]


def tournament_select(population, tournament_size=3):
    k = min(tournament_size, len(population))
    competitors = random.sample(population, k)
    winner = max(competitors, key=fitness)
    return winner.copy()


def one_point_crossover(p1, p2):
    cut = random.randint(1, N - 1)
    child1 = p1[:cut] + p2[cut:]
    child2 = p2[:cut] + p1[cut:]
    return child1, child2


def mutate(chromosome, mutation_rate=0.10):
    child = chromosome.copy()

    for i in range(N):
        if random.random() < mutation_rate:
            old = child[i]
            choices = [r for r in range(N) if r != old]
            child[i] = random.choice(choices)

    return child


def genetic_algorithm(pop_size=100, generations=500, mutation_rate=0.10, seed=0):
    random.seed(seed)

    population = [
        random_chromosome()
        for _ in range(pop_size)
    ]

    best_history = []

    for gen in range(generations):
        current_best = max(population, key=fitness)
        current_best_fitness = fitness(current_best)
        best_history.append(current_best_fitness)

        if current_best_fitness == MAX_PAIRS:
            return (
                current_best.copy(),
                current_best_fitness,
                best_history,
                gen + 1,
            )

        if gen == generations - 1:
            break

        new_population = [current_best.copy()]

        while len(new_population) < pop_size:
            p1 = tournament_select(population)
            p2 = tournament_select(population)

            c1, c2 = one_point_crossover(p1, p2)
            c1 = mutate(c1, mutation_rate)
            c2 = mutate(c2, mutation_rate)

            new_population.append(c1)

            if len(new_population) < pop_size:
                new_population.append(c2)

        population = new_population

    current_best = max(population, key=fitness)

    return (
        current_best.copy(),
        fitness(current_best),
        best_history,
        len(best_history),
    )


ga_state, ga_fit, ga_history, ga_generations = genetic_algorithm(seed=7)
print('Best chromosome:', ga_state)
print('Best fitness   :', ga_fit, '/', MAX_PAIRS)
print('Conflicts      :', MAX_PAIRS - ga_fit)
print('Generations    :', ga_generations)


Best chromosome: [3, 7, 0, 2, 5, 1, 6, 4]
Best fitness   : 28 / 28
Conflicts      : 0
Generations    : 27


In [ ]:
# take input as 
from collections import defaultdict

n, m = map(int, input().split())  # number of vertices and edges

graph = defaultdict(list)

for _ in range(m):
    u, v, w = map(int, input().split())  # edge u -> v with weight w
    graph[u].append((v, w))

    # For an undirected graph, also add:
    # graph[v].append((u, w))

In [ ]:
n = int(input())  # number of vertices
graph = []

for i in range(n):
    neighbors = list(map(int, input().split()))
    graph.append(neighbors)